In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In the previous IBM notebooks, Collaborative Filtering was done with KNN. KNN looks at a user, finds the 5 users who have similar rating histories, and averages their ratings for the target movie.

**Matrix Factorization** approaches the problem completely differently. It asks: "What if we could represent every user and every movie as a small vector of numbers (an embedding), and predict the rating by taking their dot product?"

Predicted Rating$_{u,i} = v_{user} * v_{item}$

I shall use the Python library called Surprise (pip install scikit-surprise). It is built specifically for Recommender Systems and makes implementing MF algorithms (like SVD) very clean.

First Task:
Before I can train any model, I have to prepare the data. The Surprise library doesn't take raw pandas DataFrames directly. It requires the data to be in a very specific format: a structure containing only three columns: `userID`, `itemID`, and `rating`.

In [2]:
from surprise import Dataset, SVD, Reader
import pandas as pd
#import numpy as np
import matplotlib.pyplot as plt

In [3]:
# Loading the datasets

movie_df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/BxZuF3FrO7Bdw6McwsBaBw/movies.csv')
rating_df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/R-bYYyyf7s3IUE5rsssmMw/ratings.csv')
tag_df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/UZKHhXSl7Ft7t9mfUFZJPQ/tags.csv')

In [4]:
rating_df.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [5]:
rating_df.isna().any()

userId       False
movieId      False
rating       False
timestamp    False
dtype: bool

In [6]:
rating_df.drop(columns='timestamp', inplace=True)

In [7]:
rating_df.rename(columns={'userId':'userID', 'movieId':'itemID'}, inplace=True)

In [8]:
rating_df.head()

,userID,itemID,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0


In [9]:
# A reader is still needed but only the rating_scale param is required.
min_r, max_r = rating_df.rating.min(),rating_df.rating.max() 

reader = Reader(rating_scale=(min_r, max_r))

In [10]:
# The columns must correspond to user id, item id and ratings (in that order).
data = Dataset.load_from_df(rating_df[["userID", "itemID", "rating"]], reader)

In [11]:
type(data)

surprise.dataset.DatasetAutoFolds

Splitting the dataset into train, test, split

In [12]:
from surprise.model_selection import train_test_split

train_set, test_set = train_test_split(data, test_size=0.25, random_state=42)

Training the SVD model

In [13]:
model = SVD()
model.fit(train_set)